# Q. 3 — Practical: Observe a Scan and an Index 
BritBooks wants to inspect how SQLite finds orders for one customer.
| Column        |  Data Type |
| ------------- |:-------------:|
| order_id      | INTEGER PRIMARY KEY     |
| customer_id   | INTEGER NOT NULL    |
| amount_gbp    | REAL NOT NULL     |
| status        | TEXT NOT NULL    |

TASK:
* Create query_performance.db . 
* Create the orders table. 
* Insert 5,000 sample orders. 
* Use EXPLAIN QUERY PLAN before creating an index.
* Create an index on Run customer_id. 
* EXPLAIN QUERY PLAN again. 
* Print both query plans. 

In [56]:
import sqlite3
import random 
import pandas as pd

In [ ]:
connection = sqlite3.connect("query_performance.db")
cursor = connection.cursor()
# Create and populate the table
cursor.execute("DROP TABLE IF EXISTS orders")
cursor.execute("""
    CREATE TABLE orders(
        order_id INTEGER PRIMARY KEY,
        customer_id INTEGER NOT NULL,
        amount_gbp REAL NOT NULL,
        status TEXT NOT NULL
    )
""")
import random

# Generar los datos
orders = []

for order_id in range(1, 5001):
    customer_id = random.randint(1, 1000, seed=12345)
    amount_gbp = round(random.uniform(5, 1000), 2)
    status = random.choice([
        "Pending",
        "Dispatched",
        "Confirmed",
        "Cancelled"
    ])

    orders.append((order_id, customer_id, amount_gbp, status))

cursor.executemany("""
INSERT INTO orders (order_id, customer_id, amount_gbp, status)
VALUES (?, ?, ?, ?)
""", orders)

connection.commit()

print(f"{len(orders)} registros insertados.")

query = """
SELECT order_id, amount_gbp
FROM orders
WHERE customer_id = 3
"""


5000 registros insertados.


In [58]:
# Check the plan before the index
cursor.execute(f"EXPLAIN QUERY PLAN {query}")
print("Before index:")
for row in cursor.fetchall():
    print(row)
# Check the plan after the index
#connection.close()

Before index:
(2, 0, 216, 'SCAN orders')


In [59]:
# Create the index
cursor.execute("""
CREATE INDEX idx_orders_customer_id
ON orders(customer_id)
""")
connection.commit()


In [60]:
# Check the plan after the index
cursor.execute(f"EXPLAIN QUERY PLAN {query}")
print("After index:")
for row in cursor.fetchall():
    print(row)

After index:
(3, 0, 61, 'SEARCH orders USING INDEX idx_orders_customer_id (customer_id=?)')


# Q. 4 — Practical: Filter Early 
Create an SQLite script that compares these approaches:

*Approach A:*
* Read every order. 
* Filter Dispatched rows in Python. 
* Calculate the total in Python. 

*Approach B:*
* Filter rows in SQL. 
* Calculate the total using SUM . 
* Return only the final value. 
* Print both totals and confirm that they match. 

In [ ]:
# Approach A with Pandas:
all_products_df = pd.read_sql_query("""SELECT * FROM ORDERS""", connection, index_col="order_id")
all_products_df

,customer_id,amount_gbp,status
order_id,,,
1,365,420.05,Pending
2,906,826.20,Pending
3,829,155.22,Cancelled
4,722,508.96,Cancelled
5,77,749.88,Cancelled
...,...,...,...
4996,814,42.68,Cancelled
4997,230,769.84,Cancelled
4998,386,929.12,Dispatched


In [87]:
dispatched_orders_df = all_orders_df[all_orders_df["status"]=="Dispatched"]
python_total_amount = dispatched_orders_df["amount_gbp"].sum()
python_total_amount

np.float64(650944.29)

In [ ]:
# Approach A with python standard library:
cursor.execute("""SELECT * FROM ORDERS""")
all_orders = cursor.fetchall()
all_orders

[(1, 365, 420.05, 'Pending'),
 (2, 906, 826.2, 'Pending'),
 (3, 829, 155.22, 'Cancelled'),
 (4, 722, 508.96, 'Cancelled'),
 (5, 77, 749.88, 'Cancelled'),
 (6, 252, 565.61, 'Cancelled'),
 (7, 654, 517.8, 'Pending'),
 (8, 166, 113.2, 'Confirmed'),
 (9, 831, 597.28, 'Pending'),
 (10, 170, 896.96, 'Cancelled'),
 (11, 192, 562.98, 'Pending'),
 (12, 270, 654.26, 'Pending'),
 (13, 277, 885.69, 'Cancelled'),
 (14, 199, 127.22, 'Cancelled'),
 (15, 691, 381.15, 'Dispatched'),
 (16, 781, 174.78, 'Confirmed'),
 (17, 485, 972.12, 'Pending'),
 (18, 24, 475.78, 'Dispatched'),
 (19, 158, 282.83, 'Dispatched'),
 (20, 919, 200.93, 'Cancelled'),
 (21, 529, 267.2, 'Confirmed'),
 (22, 777, 409.26, 'Dispatched'),
 (23, 90, 420.39, 'Cancelled'),
 (24, 128, 326.04, 'Dispatched'),
 (25, 88, 312.41, 'Dispatched'),
 (26, 646, 815.6, 'Dispatched'),
 (27, 567, 169.64, 'Dispatched'),
 (28, 779, 350.26, 'Confirmed'),
 (29, 806, 158.18, 'Pending'),
 (30, 233, 263.75, 'Pending'),
 (31, 577, 679.55, 'Pending'),
 (32, 3

In [ ]:
# Total calculated with for loop
total_amount=0
for  order in all_orders:
    if order[3] == "Dispatched":
        total_amount+=order[2]
total_amount

In [ ]:
# Total calculated with a generator
python_total = sum(
    amount
    for _,_, amount, status in all_orders
    if status == "Dispatched"
)

python_total

650944.29

In [ ]:
# Approach B SQL:
cursor.execute("""SELECT SUM(amount_gbp) FROM ORDERS WHERE status = 'Dispatched'""")
sql_total = cursor.fetchall()
sql_total

[(650944.29,)]

# Q. 13 — Practical: Combine Small CSV Files 
BritBooks receives three small order files for one day. 
* orders_1.csv 
* orders_2.csv 
* orders_3.csv 

Each file contains: order_id,customer_name,amount_gbp

Tasks:
* Create a folder named small_order_files . 
* Create the three CSV files with one order in each. 
* Combine them into orders_2026-07-17.csv . 
* Write the header only once. 
* Print the number of combined rows. 

In [ ]:
from pathlib import Path

In [ ]:
# Folder creation
new_folder_path = Path("Folder for session5/small_order_files")
new_folder_path.mkdir(exist_ok=True)

In [ ]:
# CSVs creation
columns_name = ["order_id", "customer_name", "amount_gbp"]
orders1 = [[1, "Vicky", 91.99]]
orders2 = [[2, "Alex", 45.45]]
orders3 = [[3, "Martin", 25]]

orders1_df = pd.DataFrame(orders1, columns=columns_name)
orders2_df = pd.DataFrame(orders2, columns=columns_name)
orders3_df = pd.DataFrame(orders3, columns=columns_name)

orders1_df.to_csv('Folder for session5/small_order_files/orders1.csv', index=False, encoding='utf-8')
orders2_df.to_csv('Folder for session5/small_order_files/orders2.csv', index=False, encoding='utf-8')
orders3_df.to_csv('Folder for session5/small_order_files/orders3.csv', index=False, encoding='utf-8')


In [ ]:
# CSVs read
o1_read=pd.read_csv('Folder for session5/small_order_files/orders1.csv', header=0)
o2_read=pd.read_csv('Folder for session5/small_order_files/orders2.csv', header=0)
o3_read=pd.read_csv('Folder for session5/small_order_files/orders3.csv', header=0)

pandas.DataFrame

In [ ]:
# Combine the CSVs
combined_df = pd.concat([o3_read, o2_read, o3_read])
combined_df

,order_id,customer_name,amount_gbp
0,3,Martin,25.00
0,2,Alex,45.45
0,3,Martin,25.00


In [ ]:
#create combined CSV
combined_df.to_csv('Folder for session5/small_order_files/orders_2026-07-17.csv', index=False, encoding='utf-8')

# Q. 22 — Practical: Create an Avro-Style Schema File 
BritBooks sends order events between systems. 

REQUIRED FIELDS:
| Field         |  Type |
| ------------- |:-------------:|
| order_id      | string     |
| customer_id   | string    |
| amount_gbp    | double     |
| order_status  | string    |
| coupon_code   | null or string, default null| 

TASK:

* Create the schema as a Python dictionary.
* Write it to order_event_schema.json . 
* Use indentation to keep the JSON readable. 
* Read the file again. 
* Print the field names. 

In [113]:
import json

In [ ]:
# Schema aspython dictionary
schema = {
    "name": "order_events",
    "type": "record",
    "fields": [
        {
            "name": "order_id", 
            "type": "string"
        },
        {
            "name": "customer_id", 
            "type": "string"
        },
        {
            "name": "amount_gbp", 
            "type": "double"
        },
        {
            "name": "order_status", 
            "type": "string"
        },
        {
            "name": "customer_id", 
            "type": "string"
        },
        {
            "name": "coupon_code", 
            "type": ["string", "null"],
            "default": None
        }
    ]
}

In [119]:
# Write schema into a json
with open("Folder for session5/order_event_schema.json", "w", encoding="utf-8") as file:
    json.dump(schema, file, indent=4)

In [ ]:
# Read the json
with open("Folder for session5/order_event_schema.json", "r", encoding="utf-8") as file:
    json_schema_file = json.load(file)

json_schema_file

{'name': 'order_events',
 'type': 'record',
 'fields': [{'name': 'order_id', 'type': 'string'},
  {'name': 'customer_id', 'type': 'string'},
  {'name': 'amount_gbp', 'type': 'double'},
  {'name': 'order_status', 'type': 'string'},
  {'name': 'customer_id', 'type': 'string'},
  {'name': 'coupon_code', 'type': ['string', 'null'], 'default': None}]}

In [ ]:
# Read the files
json_schema_fields = [field["name"] for field in json_schema_file["fields"]]
json_schema_fields

['order_id',
 'customer_id',
 'amount_gbp',
 'order_status',
 'customer_id',
 'coupon_code']

# Q. 30 — Practical: Create a Partitioned Folder Layout 
BritBooks wants to organise orders by year and month. 

_1001, 2026-06-28, £34.99_

_1002, 2026-07-02, £18.50_

_1003, 2026-07-17, £42.75_

* Create an orders_lake folder. 
* Create year=YYYY/month=MM folders from each date. 
* Write each order to an orders.csv file in the correct partition. 
* Include the CSV header. 
* Print the created file paths.

In [ ]:
from datetime import datetime
import csv

In [ ]:
# Orders lake folder creation
order_lake_path = Path("Folder for session5/order_lake")
order_lake_path.mkdir(exist_ok=True)

In [121]:
orders = [
    {"order_id": "1001", "order_date": "2026-06-28", "amount_gbp": 34.99},
    {"order_id": "1002", "order_date": "2026-07-02", "amount_gbp": 18.50},
    {"order_id": "1003", "order_date": "2026-07-17", "amount_gbp": 42.75}
]

In [ ]:
# Loop for creating directories, csv file and the data
for order in orders:
    date = datetime.strptime(order["order_date"], "%Y-%m-%d").date()
    header = order.keys()

    folderpath_year_month = (
        order_lake_path / 
        f"{date.strftime('%Y')}" /
        f"{date.strftime('%m')}" 
    )
    folderpath_year_month.mkdir(exist_ok=True, parents=True)

    csv_path = folderpath_year_month / "orders.csv"

    if csv_path.exists():
        mode = "a"
    else:
        mode = "w"

    with open(csv_path, mode , encoding="utf-8") as file:
        writer = csv.writer(file, lineterminator='\n')
        if mode=="w":
            writer.writerow(order.keys())
        writer.writerow(order.values())
    
     

In [163]:
for file_path in sorted(order_lake_path.rglob("orders.csv")):
    print(file_path)

Folder for session5\order_lake\2026\06\orders.csv
Folder for session5\order_lake\2026\07\orders.csv
